# Building a Lightweight RAG System for Academic Papers

**Audience**
- Students building an applied AI course project around local PDF question answering.

**Prerequisites**
- Basic Python, pandas, and vector search concepts.
- A folder of local academic papers in PDF format.

**Learning goals**
- Parse academic PDFs into structured text.
- Build a lightweight retrieval pipeline with chunking, embeddings, and vector search.
- Generate grounded answers with citations.
- Run a small ablation study so the notebook contains analysis rather than only a demo.

This notebook implements a compact RAG pipeline for the papers stored in `data/`. The design is intentionally pragmatic for a course project: it works with a small collection, keeps each step transparent, and includes a mini retrieval evaluation section that can be reused in the final report.


## Outline

1. Check the environment and locate the paper collection.
2. Extract text from local PDFs.
3. Split the papers into overlapping chunks.
4. Build embeddings and a vector index.
5. Retrieve evidence for a user question.
6. Produce a grounded answer with citations.
7. Compare a few retrieval settings in a small ablation study.
8. Summarize pitfalls, extensions, and exercises.


In [ ]:
from __future__ import annotations

import importlib.util
import os
import random
import re
import textwrap
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

SEED = 5201
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
PDF_PATHS = sorted(DATA_DIR.glob("*.pdf"))

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)

assert PDF_PATHS, f"No PDF files were found in {DATA_DIR}"
[path.name for path in PDF_PATHS]


In [ ]:
def package_available(package_name: str) -> bool:
    return importlib.util.find_spec(package_name) is not None


package_status = pd.DataFrame(
    {
        "package": [
            "pypdf",
            "numpy",
            "pandas",
            "scikit-learn",
            "faiss",
            "sentence_transformers",
            "transformers",
            "torch",
            "openai",
        ],
        "available": [
            True,
            True,
            True,
            True,
            package_available("faiss"),
            package_available("sentence_transformers"),
            package_available("transformers"),
            package_available("torch"),
            package_available("openai"),
        ],
        "role": [
            "PDF parsing",
            "Numerical operations",
            "Tables and analysis",
            "Vectorization and fallback retrieval",
            "Primary vector database",
            "Preferred lightweight embedding model",
            "Optional local generator",
            "Backend for local generator",
            "Qwen API client",
        ],
    }
)

package_status


The notebook is written to be robust on limited machines.

- If `faiss` is available, the index uses FAISS as required by the project topic.
- If `sentence_transformers` is available, the notebook uses `all-MiniLM-L6-v2` for stronger semantic retrieval.
- If those packages are missing, the code falls back to TF-IDF plus `NearestNeighbors`, so the full pipeline is still runnable for debugging and demonstration.
- If `transformers` is installed, you can switch on a local text generation step; otherwise the notebook uses an extractive answer composer.

If you need the full stack, install the missing packages in a notebook cell such as:

```python
# %pip install pypdf faiss-cpu sentence-transformers transformers torch openai scikit-learn matplotlib pandas
```


## Qwen API Configuration

The notebook can also call a hosted Qwen model through DashScope's OpenAI-compatible API. For safety, the API key is read from environment variables or from a local `.env` file that is ignored by Git.


In [ ]:
def load_env_file(env_path: Path = BASE_DIR / ".env") -> None:
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_env_file()

QWEN_API_KEY = os.getenv("DASHSCOPE_API_KEY", "")
QWEN_BASE_URL = os.getenv("DASHSCOPE_BASE_URL", "https://dashscope-intl.aliyuncs.com/compatible-mode/v1")
QWEN_MODEL = os.getenv("QWEN_MODEL", "qwen-plus")

{
    "qwen_api_configured": bool(QWEN_API_KEY),
    "qwen_base_url": QWEN_BASE_URL,
    "qwen_model": QWEN_MODEL,
}


## Step 1. Read the local academic papers

We first convert each PDF into page-level text records. Page-level storage is useful because it gives us a simple citation target later, such as a paper title plus a page number.


In [ ]:
def normalize_line(line: str) -> str:
    """Normalise whitespace within a single line (no newline collapsing)."""
    line = line.replace("\x00", " ")
    line = re.sub(r"[ \t]+", " ", line)   # collapse spaces/tabs, keep newline intact
    return line.strip()


def clean_academic_text(raw_text: str, keep_tables: bool = False) -> str:
    """
    Clean page-level text extracted by pypdf.

    Must receive the RAW text (before any newline collapsing) so that
    line-level filters can operate correctly.

    Parameters
    ----------
    raw_text : str
        Text as returned by ``page.extract_text()``, with original newlines.
    keep_tables : bool
        False (default) — filter out linearised result-table rows.
                          Best for source-lookup / explanatory questions.
        True            — retain numeric table rows so factual queries
                          (e.g. "what accuracy did method X achieve?")
                          can match specific experimental numbers.

    Filters always applied (regardless of keep_tables):
      - Isolated formula fragments  (symbol-heavy, <4 prose words)
      - Reference-section lines     ([1] Author … 2023, pages X–Y)
      - Non-ASCII-dense lines       (garbled LaTeX / non-target-language)
      - Very short stub lines       (<4 chars, layout artefacts)

    Filter controlled by keep_tables:
      - Pure numeric / table rows   (≥60 % digits+punctuation)
    """
    ref_pattern = re.compile(
        r"^\s*(\[\d+\]|\d+\.)\s+\w.*\d{4}"
        r"|pages?\s+\d+[–\-]\d+"
        r"|\b(arxiv|doi|http|isbn)\b",
        re.IGNORECASE,
    )
    formula_op = re.compile(r"[=<>∑∫∂→←≤≥≠≈∈⊆∪∩√∞±×÷\|]|\\[a-z]+\{")

    def is_table_row(line: str) -> bool:
        core = line.replace(" ", "")
        if len(core) < 6:
            return False
        digit_punct = sum(1 for c in core if c.isdigit() or c in "±.,/%()[]−-+×·")
        return digit_punct / len(core) >= 0.60

    def is_formula_fragment(line: str) -> bool:
        words = re.findall(r"[A-Za-z]{2,}", line)
        return len(words) < 4 and bool(formula_op.search(line))

    def is_non_ascii_dense(line: str) -> bool:
        if len(line) < 6:
            return False
        non_ascii = sum(1 for c in line if ord(c) > 127)
        return non_ascii / len(line) > 0.35

    kept: list[str] = []
    for raw_line in raw_text.splitlines():
        # Fix hyphenated line-break at the tail of a line before filtering
        line = re.sub(r"(\w)-$", r"\1", raw_line)   # "infor-" → "infor" (joins next line)
        line = normalize_line(line)
        if not line or len(line) < 4:
            continue
        if ref_pattern.search(line):
            continue
        if is_non_ascii_dense(line):
            continue
        if is_formula_fragment(line):
            continue
        if not keep_tables and is_table_row(line):
            continue
        kept.append(line)

    # Join surviving lines; collapse residual internal whitespace
    joined = " ".join(kept)
    return re.sub(r"\s+", " ", joined).strip()


# ── Cleaning mode ─────────────────────────────────────────────────────────────
# False → filter table rows  (better for source-lookup / explanatory queries)
# True  → keep table rows    (better for factual queries about specific numbers)
KEEP_TABLES = False


def load_pdf_pages(pdf_path: Path) -> list[dict[str, Any]]:
    reader = PdfReader(str(pdf_path))
    rows: list[dict[str, Any]] = []
    for page_number, page in enumerate(reader.pages, start=1):
        raw  = page.extract_text() or ""          # keep original newlines
        text = clean_academic_text(raw, keep_tables=KEEP_TABLES)
        if not text:
            continue
        rows.append(
            {
                "paper_id":    pdf_path.stem,
                "paper_title": pdf_path.stem,
                "source_file": pdf_path.name,
                "page_number": page_number,
                "text":        text,
                "char_count":  len(text),
                "word_count":  len(text.split()),
            }
        )
    return rows


page_rows: list[dict[str, Any]] = []
for pdf_path in PDF_PATHS:
    page_rows.extend(load_pdf_pages(pdf_path))

pages_df = pd.DataFrame(page_rows)

print(f"Cleaning mode : keep_tables={KEEP_TABLES}")
audit = (
    pages_df.groupby("paper_title")["word_count"]
    .sum()
    .reset_index()
    .rename(columns={"paper_title": "paper", "word_count": "words_after_cleaning"})
)
print(audit.to_string(index=False))
print(f"\nTotal pages retained : {len(pages_df)}")


In [ ]:
papers_df = (
    pages_df.groupby(["paper_id", "paper_title", "source_file"], as_index=False)
    .agg(
        pages=("page_number", "nunique"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
    )
    .sort_values("paper_title")
    .reset_index(drop=True)
)

papers_df


The collection is intentionally small, which is a good fit for a lightweight RAG system. Instead of optimizing for distributed scale, we optimize for transparency and reproducibility.


## Step 2. Split each page into overlapping chunks

Retrieval quality depends heavily on chunking. Chunks that are too short lose context, while chunks that are too long dilute the signal and may retrieve irrelevant content. A simple and effective baseline is overlapping word windows.


In [ ]:
def chunk_text(text: str, chunk_size: int = 180, overlap: int = 40) -> list[str]:
    words = text.split()
    if not words:
        return []

    step = max(chunk_size - overlap, 1)
    chunks: list[str] = []
    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks


def build_chunks(page_table: pd.DataFrame, chunk_size: int = 180, overlap: int = 40) -> pd.DataFrame:
    chunk_rows: list[dict[str, Any]] = []
    chunk_id = 0

    for row in page_table.itertuples(index=False):
        page_chunks = chunk_text(row.text, chunk_size=chunk_size, overlap=overlap)
        for local_chunk_id, chunk in enumerate(page_chunks, start=1):
            chunk_rows.append(
                {
                    "chunk_id": chunk_id,
                    "paper_id": row.paper_id,
                    "paper_title": row.paper_title,
                    "source_file": row.source_file,
                    "page_number": row.page_number,
                    "local_chunk_id": local_chunk_id,
                    "chunk_text": chunk,
                    "chunk_word_count": len(chunk.split()),
                    "citation": f"{row.paper_title}, p.{row.page_number}",
                }
            )
            chunk_id += 1

    return pd.DataFrame(chunk_rows)


DEFAULT_CHUNK_SIZE = 180
DEFAULT_OVERLAP = 40
chunks_df = build_chunks(pages_df, chunk_size=DEFAULT_CHUNK_SIZE, overlap=DEFAULT_OVERLAP)

chunks_df[["paper_title", "page_number", "local_chunk_id", "chunk_word_count", "citation"]].head(10)


In [ ]:
chunk_summary = (
    chunks_df.groupby("paper_title", as_index=False)
    .agg(
        chunk_count=("chunk_id", "count"),
        avg_chunk_words=("chunk_word_count", "mean"),
    )
    .sort_values("chunk_count", ascending=False)
)

chunk_summary


In [ ]:
sample_chunk = chunks_df.iloc[0]
print(sample_chunk["citation"])
print()
print(textwrap.fill(sample_chunk["chunk_text"], width=100))


## Step 3. Build embeddings and the vector index

We support two embedding backends:

- `MiniLM` if `sentence_transformers` is available. This is the preferred semantic encoder for a stronger project demo.
- `TF-IDF` as a lightweight fallback that avoids model downloads and still lets the entire workflow run.

We also support two retrieval engines:

- `FAISS` if installed.
- `NearestNeighbors` as a local fallback.


In [ ]:
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    matrix = matrix.astype("float32")
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


# Local model path — avoids HuggingFace network dependency
LOCAL_MINILM_PATH = str(BASE_DIR / "models" / "all-MiniLM-L6-v2")


def build_embedding_backend(texts: list[str], preference: str = "auto") -> dict[str, Any]:
    sentence_transformers_ready = package_available("sentence_transformers")

    if preference in {"auto", "minilm"} and sentence_transformers_ready:
        try:
            from sentence_transformers import SentenceTransformer

            # Try local path first; fall back to hub name if local copy is missing
            model_name = LOCAL_MINILM_PATH if Path(LOCAL_MINILM_PATH).exists() else "sentence-transformers/all-MiniLM-L6-v2"
            model = SentenceTransformer(model_name)
            embeddings = model.encode(
                texts,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            ).astype("float32")
            return {
                "backend": "MiniLM",
                "model_name": model_name,
                "encoder": model,
                "vectorizer": None,
                "embeddings": embeddings,
            }
        except Exception as exc:
            print(f"MiniLM backend unavailable ({exc}). Falling back to TF-IDF.")

    vectorizer = TfidfVectorizer(max_features=4096, ngram_range=(1, 2), stop_words="english")
    embeddings = vectorizer.fit_transform(texts).toarray().astype("float32")
    embeddings = l2_normalize(embeddings)
    return {
        "backend": "TF-IDF",
        "model_name": "sklearn_tfidf_bigram",
        "encoder": None,
        "vectorizer": vectorizer,
        "embeddings": embeddings,
    }


def encode_queries(queries: list[str], backend_bundle: dict[str, Any]) -> np.ndarray:
    if backend_bundle["backend"] == "MiniLM":
        return backend_bundle["encoder"].encode(
            queries,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype("float32")

    query_matrix = backend_bundle["vectorizer"].transform(queries).toarray().astype("float32")
    return l2_normalize(query_matrix)


In [ ]:
def build_retriever(embeddings: np.ndarray) -> dict[str, Any]:
    if package_available("faiss"):
        import faiss

        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings.astype("float32"))
        return {"engine": "FAISS", "index": index}

    fallback = NearestNeighbors(metric="cosine", algorithm="brute")
    fallback.fit(embeddings)
    return {"engine": "NearestNeighbors", "index": fallback}


embedding_bundle = build_embedding_backend(chunks_df["chunk_text"].tolist(), preference="auto")
retriever_bundle = build_retriever(embedding_bundle["embeddings"])

{
    "embedding_backend": embedding_bundle["backend"],
    "embedding_model": embedding_bundle["model_name"],
    "retrieval_engine": retriever_bundle["engine"],
    "num_chunks": len(chunks_df),
    "embedding_dim": int(embedding_bundle["embeddings"].shape[1]),
}


In [ ]:
## Index Persistence
# Serialise the chunk table and embeddings so repeated runs skip re-encoding.
# Delete the .rag_cache/ folder whenever PDFs change.

CACHE_DIR = BASE_DIR / ".rag_cache"
CACHE_DIR.mkdir(exist_ok=True)


def save_index_to_disk(
    chunks: pd.DataFrame,
    embeddings: np.ndarray,
    backend_name: str,
) -> None:
    """Save chunk metadata (JSON) and embeddings (.npz) to CACHE_DIR."""
    np.savez_compressed(
        CACHE_DIR / f"embeddings_{backend_name}.npz",
        embeddings=embeddings.astype("float32"),
    )
    chunks.to_json(
        CACHE_DIR / "chunks.json",
        orient="records",
        force_ascii=False,
        indent=2,
    )
    print(
        f"Index saved → {CACHE_DIR}  "
        f"({len(chunks)} chunks, shape={embeddings.shape}, backend={backend_name})"
    )


def load_index_from_disk(backend_name: str) -> tuple[pd.DataFrame, np.ndarray] | None:
    """Return (chunks_df, embeddings) from cache, or None if cache is missing."""
    emb_path   = CACHE_DIR / f"embeddings_{backend_name}.npz"
    chunk_path = CACHE_DIR / "chunks.json"
    if not emb_path.exists() or not chunk_path.exists():
        return None
    embeddings = np.load(emb_path)["embeddings"]
    chunks     = pd.read_json(chunk_path)
    print(
        f"Index loaded from cache  "
        f"({len(chunks)} chunks, shape={embeddings.shape}, backend={backend_name})"
    )
    return chunks, embeddings


# ── Try cache first; fall back to the freshly built index ────────────────────
_backend_key = embedding_bundle["backend"]
_cached = load_index_from_disk(_backend_key)

if _cached is not None:
    chunks_df, _cached_embeddings = _cached
    retriever_bundle = build_retriever(_cached_embeddings)
    print("Retriever rebuilt from cached embeddings.")
else:
    save_index_to_disk(chunks_df, embedding_bundle["embeddings"], _backend_key)
    print("Fresh index saved to cache.")


## Step 4. Retrieve relevant evidence for a user question

The retrieval step is where the RAG system earns its usefulness. If retrieval is poor, answer generation will also be poor, even if the language model is strong.


In [ ]:
def search_chunks(
    question: str,
    chunks_table: pd.DataFrame,
    embedding_backend: dict[str, Any],
    retriever: dict[str, Any],
    top_k: int = 4,
) -> pd.DataFrame:
    query_vector = encode_queries([question], embedding_backend)

    if retriever["engine"] == "FAISS":
        scores, indices = retriever["index"].search(query_vector.astype("float32"), top_k)
        ranked_indices = indices[0].tolist()
        ranked_scores = scores[0].tolist()
    else:
        distances, indices = retriever["index"].kneighbors(query_vector, n_neighbors=top_k)
        ranked_indices = indices[0].tolist()
        ranked_scores = (1 - distances[0]).tolist()

    result = chunks_table.iloc[ranked_indices].copy().reset_index(drop=True)
    result.insert(0, "rank", range(1, len(result) + 1))
    result.insert(1, "score", ranked_scores)
    result["preview"] = result["chunk_text"].apply(lambda x: textwrap.shorten(x, width=180, placeholder="..."))
    return result


demo_question = "How does multi-agent debate improve factuality and reasoning in language models?"
demo_results = search_chunks(
    demo_question,
    chunks_table=chunks_df,
    embedding_backend=embedding_bundle,
    retriever=retriever_bundle,
    top_k=4,
)

demo_results[["rank", "score", "paper_title", "page_number", "citation", "preview"]]


Retrieved chunks should already look interpretable. This is important for a course project because you can show these ranked passages directly in the report or demo to explain why the final answer is grounded.


## Step 5. Turn retrieved evidence into an answer

This notebook supports two answer modes:

- `extractive`: always available and easy to run.
- `qwen_api`: uses a hosted Qwen model through DashScope's OpenAI-compatible API.
- `local_llm`: optional, uses a small local text generation model when `transformers` and `torch` are installed.

The extractive mode is still useful because it produces evidence-backed answers with citations and avoids a hard dependency on a large model during development.


In [ ]:
def sentence_split(text: str) -> list[str]:
    pieces = re.split(r"(?<=[.!?])\s+", text)
    return [piece.strip() for piece in pieces if piece.strip()]


def keyword_set(question: str) -> set[str]:
    tokens = re.findall(r"[A-Za-z]{3,}", question.lower())
    stopwords = {
        "what",
        "when",
        "where",
        "which",
        "their",
        "there",
        "about",
        "would",
        "could",
        "should",
        "does",
        "into",
        "from",
        "that",
        "with",
        "this",
        "have",
        "been",
        "using",
        "through",
    }
    return {token for token in tokens if token not in stopwords}


def sentence_is_clean(sentence: str) -> bool:
    words = sentence.split()
    if len(words) < 8 or len(words) > 80:
        return False

    lowered = sentence.lower()
    noisy_markers = ["figure", "table", "copyright", "proceedings", "http", "arxiv"]
    if any(marker in lowered for marker in noisy_markers):
        return False

    alpha_chars = sum(char.isalpha() for char in sentence)
    total_chars = len(sentence.replace(" ", ""))
    if total_chars == 0:
        return False

    alpha_ratio = alpha_chars / total_chars
    return alpha_ratio >= 0.65


def build_extractive_answer(question: str, retrieved_chunks: pd.DataFrame, max_sentences: int = 3) -> str:
    query_terms = keyword_set(question)
    candidates: list[tuple[float, str]] = []

    for row in retrieved_chunks.itertuples(index=False):
        for sentence in sentence_split(row.chunk_text):
            if not sentence_is_clean(sentence):
                continue
            lowered = sentence.lower()
            overlap = sum(term in lowered for term in query_terms)
            if overlap == 0:
                continue
            score = float(row.score) + 0.15 * overlap
            citation = f"[{row.citation}]"
            candidates.append((score, f"{sentence} {citation}"))

    ranked = sorted(candidates, key=lambda item: item[0], reverse=True)
    selected: list[str] = []
    for _, sentence in ranked:
        if sentence not in selected:
            selected.append(sentence)
        if len(selected) >= max_sentences:
            break

    if not selected:
        fallback = retrieved_chunks.iloc[0]
        selected.append(f"{fallback.chunk_text[:240].strip()}... [{fallback.citation}]")

    return " ".join(selected)


def build_context_block(retrieved_chunks: pd.DataFrame) -> str:
    blocks = []
    for row in retrieved_chunks.itertuples(index=False):
        blocks.append(f"Source: {row.citation}\nText: {row.chunk_text}")
    return "\n\n".join(blocks)


def answer_question(
    question: str,
    retrieved_chunks: pd.DataFrame,
    mode: str = "extractive",
    local_model_name: str = "google/flan-t5-small",
) -> tuple[str, str]:
    if mode == "qwen_api":
        if not package_available("openai"):
            raise ImportError("Install the openai package to use qwen_api mode.")
        if not QWEN_API_KEY:
            raise ValueError("DASHSCOPE_API_KEY is missing. Add it to .env or environment variables.")

        from openai import OpenAI

        client = OpenAI(api_key=QWEN_API_KEY, base_url=QWEN_BASE_URL)
        completion = client.chat.completions.create(
            model=QWEN_MODEL,
            temperature=0.1,
            max_tokens=400,
            messages=[
                {
                    "role": "system",
                    "content": "You answer questions about academic papers using only the retrieved evidence. Cite sources inline. If the evidence is insufficient, say so clearly.",
                },
                {
                    "role": "user",
                    "content": f"Question: {question}\n\nEvidence:\n{build_context_block(retrieved_chunks)}",
                },
            ],
        )
        return completion.choices[0].message.content.strip(), "qwen_api"

    if mode == "local_llm" and package_available("transformers") and package_available("torch"):
        from transformers import pipeline

        generator = pipeline("text2text-generation", model=local_model_name)
        prompt = f'''
        Answer the question only with the evidence below.
        If the evidence is insufficient, say so clearly.

        Question: {question}

        Evidence:
        {build_context_block(retrieved_chunks)}

        Give a concise answer with inline citations.
        '''.strip()

        response = generator(prompt, max_new_tokens=220, do_sample=False)[0]["generated_text"].strip()
        return response, "local_llm"

    return build_extractive_answer(question, retrieved_chunks), "extractive"


In [ ]:
answer_text, answer_mode = answer_question(demo_question, demo_results, mode="extractive")

print("Answer mode:", answer_mode)
print()
print(textwrap.fill(answer_text, width=110))


A practical demo function is useful in the final presentation because it lets you show the full chain from question to evidence to answer.


In [ ]:
def run_rag_demo(question: str, top_k: int = 4, answer_mode: str = "extractive") -> dict[str, Any]:
    retrieved = search_chunks(
        question,
        chunks_table=chunks_df,
        embedding_backend=embedding_bundle,
        retriever=retriever_bundle,
        top_k=top_k,
    )
    answer, used_mode = answer_question(question, retrieved, mode=answer_mode)
    return {
        "question": question,
        "answer_mode": used_mode,
        "answer": answer,
        "retrieved_chunks": retrieved[["rank", "score", "citation", "preview"]],
    }


demo = run_rag_demo("Which paper argues that sparse communication can reduce computation cost in multi-agent debate?")
print("Question:", demo["question"])
print("Mode:", demo["answer_mode"])
print()
print(textwrap.fill(demo["answer"], width=110))
print()
demo["retrieved_chunks"]


## Step 6. Add a small ablation study

The project instructions explicitly ask for new insights rather than only running existing code. A simple way to contribute something meaningful is to compare retrieval settings and report which configuration retrieves the correct paper more reliably.


In [ ]:
evaluation_questions = pd.DataFrame(
    [
        # ── Source-lookup: identify which paper covers a topic ──────────────────
        {
            "question": "Which paper reports that debate reduces fallacious answers and hallucinations?",
            "expected_paper": "Improving Factuality and Reasoning in Language Models through Multiagent Debate",
            "type": "source_lookup",
        },
        {
            "question": "Which paper introduces the Explain-Analyze-Generate sequential collaboration method?",
            "expected_paper": "Explain-Analyze-Generate A Sequential Multi-Agent Collaboration Method for Complex Reasoning",
            "type": "source_lookup",
        },
        {
            "question": "Which paper studies sparse communication topology in multi-agent debate?",
            "expected_paper": "Improving Multi-Agent Debate with Sparse Communication Topology",
            "type": "source_lookup",
        },
        {
            "question": "Which paper discusses degeneration-of-thought and divergent thinking in debate?",
            "expected_paper": "Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate",
            "type": "source_lookup",
        },
        {
            "question": "Which paper extends debate ideas to multimodal reasoning and alignment labeling tasks?",
            "expected_paper": "Improving Multi-Agent Debate with Sparse Communication Topology",
            "type": "source_lookup",
        },
        {
            "question": "Which paper compares debate with self-reflection-style methods for complex reasoning?",
            "expected_paper": "Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate",
            "type": "source_lookup",
        },
        # ── Factual: retrieve specific facts stated in the paper ────────────────
        {
            "question": "What reasoning benchmarks are used to evaluate the multiagent debate approach?",
            "expected_paper": "Improving Factuality and Reasoning in Language Models through Multiagent Debate",
            "type": "factual",
        },
        {
            "question": "What are the three sequential stages defined in the EAG collaboration framework?",
            "expected_paper": "Explain-Analyze-Generate A Sequential Multi-Agent Collaboration Method for Complex Reasoning",
            "type": "factual",
        },
        {
            "question": "What graph structures are compared for agent communication topology in multi-agent debate?",
            "expected_paper": "Improving Multi-Agent Debate with Sparse Communication Topology",
            "type": "factual",
        },
        {
            "question": "What failure mode is identified when agents unconditionally follow the majority opinion in debate?",
            "expected_paper": "Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate",
            "type": "factual",
        },
        # ── Comparative: compare methods or results across conditions ───────────
        {
            "question": "How does the accuracy of multiagent debate compare to self-consistency on math reasoning tasks?",
            "expected_paper": "Improving Factuality and Reasoning in Language Models through Multiagent Debate",
            "type": "comparative",
        },
        {
            "question": "How does EAG performance compare to single-agent and standard round-based debate baselines?",
            "expected_paper": "Explain-Analyze-Generate A Sequential Multi-Agent Collaboration Method for Complex Reasoning",
            "type": "comparative",
        },
        {
            "question": "Does sparse topology achieve better performance than fully connected debate with fewer communication tokens?",
            "expected_paper": "Improving Multi-Agent Debate with Sparse Communication Topology",
            "type": "comparative",
        },
        {
            "question": "How does divergent-thinking debate compare to standard debate on common-sense reasoning benchmarks?",
            "expected_paper": "Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate",
            "type": "comparative",
        },
        # ── Explanatory: require understanding of mechanisms or reasoning ───────
        {
            "question": "Why does having multiple agents debate each other reduce hallucination in language model outputs?",
            "expected_paper": "Improving Factuality and Reasoning in Language Models through Multiagent Debate",
            "type": "explanatory",
        },
        {
            "question": "Why does the explain phase improve the quality of subsequent analysis in the EAG method?",
            "expected_paper": "Explain-Analyze-Generate A Sequential Multi-Agent Collaboration Method for Complex Reasoning",
            "type": "explanatory",
        },
        {
            "question": "Why can sparse communication topology maintain debate quality while reducing communication overhead?",
            "expected_paper": "Improving Multi-Agent Debate with Sparse Communication Topology",
            "type": "explanatory",
        },
        {
            "question": "Why does degeneration-of-thought occur and how does divergent initialization prevent it?",
            "expected_paper": "Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate",
            "type": "explanatory",
        },
    ]
)

print(f"Total evaluation questions: {len(evaluation_questions)}")
evaluation_questions.groupby("type").size().rename("count").to_frame()


In [ ]:
def retrieval_rank(
    question: str,
    expected_paper: str,
    chunk_size: int,
    overlap: int,
    top_k: int,
    embedding_preference: str = "auto",
) -> int | None:
    candidate_chunks = build_chunks(pages_df, chunk_size=chunk_size, overlap=overlap)
    candidate_backend = build_embedding_backend(candidate_chunks["chunk_text"].tolist(), preference=embedding_preference)
    candidate_retriever = build_retriever(candidate_backend["embeddings"])
    results = search_chunks(
        question,
        chunks_table=candidate_chunks,
        embedding_backend=candidate_backend,
        retriever=candidate_retriever,
        top_k=top_k,
    )

    for row in results.itertuples(index=False):
        if expected_paper.lower() in row.paper_title.lower():
            return int(row.rank)
    return None


def evaluate_configuration(
    chunk_size: int,
    overlap: int,
    top_k: int,
    embedding_preference: str = "auto",
) -> dict[str, Any]:
    ranks: list[int | None] = []
    for row in evaluation_questions.itertuples(index=False):
        rank = retrieval_rank(
            question=row.question,
            expected_paper=row.expected_paper,
            chunk_size=chunk_size,
            overlap=overlap,
            top_k=top_k,
            embedding_preference=embedding_preference,
        )
        ranks.append(rank)

    hit_rate = np.mean([rank is not None for rank in ranks])
    top1_accuracy = np.mean([rank == 1 for rank in ranks])
    mrr = np.mean([0 if rank is None else 1 / rank for rank in ranks])

    return {
        "chunk_size": chunk_size,
        "overlap": overlap,
        "top_k": top_k,
        "embedding_backend": "MiniLM if available else TF-IDF",
        "hit_rate": round(float(hit_rate), 3),
        "top1_accuracy": round(float(top1_accuracy), 3),
        "mrr": round(float(mrr), 3),
    }


candidate_settings = [
    {"chunk_size": 120, "overlap": 20, "top_k": 3},
    {"chunk_size": 180, "overlap": 40, "top_k": 3},
    {"chunk_size": 220, "overlap": 50, "top_k": 4},
    {"chunk_size": 260, "overlap": 60, "top_k": 5},
]

ablation_results = pd.DataFrame(
    [evaluate_configuration(**setting) for setting in candidate_settings]
).sort_values(["mrr", "top1_accuracy", "hit_rate"], ascending=False).reset_index(drop=True)

ablation_results


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(ablation_results.index.astype(str), ablation_results["mrr"], color="#3A6EA5")
plt.xticks(ablation_results.index.astype(str))
plt.xlabel("Configuration rank")
plt.ylabel("MRR")
plt.title("Retrieval Ablation Across Chunking Configurations")
plt.ylim(0, 1.05)
plt.show()

ablation_results


In [ ]:
## Step 6b. Compare retrieval backends (Table 2)
# Fix chunk_size=220/overlap=50 (best from ablation) and compare
# MiniLM+FAISS against TF-IDF+NearestNeighbors across all question types.

def evaluate_backend_comparison(
    chunk_size: int = 220,
    overlap: int = 50,
    top_k: int = 3,
) -> pd.DataFrame:
    """Return a DataFrame with retrieval metrics for each backend."""
    candidate_chunks = build_chunks(pages_df, chunk_size=chunk_size, overlap=overlap)
    rows = []

    for preference, label in [("minilm", "MiniLM + FAISS"), ("tfidf", "TF-IDF + NN")]:
        candidate_backend   = build_embedding_backend(
            candidate_chunks["chunk_text"].tolist(), preference=preference
        )
        candidate_retriever = build_retriever(candidate_backend["embeddings"])

        ranks_all: list[int | None] = []
        per_type: dict[str, list[int | None]] = {}

        for row in evaluation_questions.itertuples(index=False):
            results = search_chunks(
                row.question,
                chunks_table=candidate_chunks,
                embedding_backend=candidate_backend,
                retriever=candidate_retriever,
                top_k=top_k,
            )
            rank = next(
                (int(r.rank) for r in results.itertuples()
                 if row.expected_paper.lower() in r.paper_title.lower()),
                None,
            )
            ranks_all.append(rank)
            per_type.setdefault(row.type, []).append(rank)

        def metrics(ranks: list) -> dict:
            return {
                "top1_accuracy": round(float(np.mean([r == 1 for r in ranks])), 3),
                f"hit_rate@{top_k}": round(float(np.mean([r is not None for r in ranks])), 3),
                "mrr":           round(float(np.mean([0 if r is None else 1 / r for r in ranks])), 3),
            }

        row_data: dict = {"backend": label, "engine": candidate_retriever["engine"]}
        row_data.update(metrics(ranks_all))
        for qtype, type_ranks in per_type.items():
            row_data[f"mrr_{qtype}"] = round(
                float(np.mean([0 if r is None else 1 / r for r in type_ranks])), 3
            )
        rows.append(row_data)

    return pd.DataFrame(rows)


print("Running backend comparison (chunk_size=220/50, k=3) …")
backend_df = evaluate_backend_comparison()

# ── Summary chart ─────────────────────────────────────────────────────────────
metrics_to_plot = ["top1_accuracy", "hit_rate@3", "mrr"]
titles_to_plot  = ["Top-1 Accuracy", "Hit Rate@3", "MRR"]
bar_colors      = ["#3A6EA5", "#E07B39"]
x = np.arange(len(metrics_to_plot))
width = 0.3

fig, ax = plt.subplots(figsize=(8, 4))
for i, (_, brow) in enumerate(backend_df.iterrows()):
    vals = [brow[m] for m in metrics_to_plot]
    ax.bar(x + i * width, vals, width, label=brow["backend"], color=bar_colors[i])

ax.set_xticks(x + width / 2)
ax.set_xticklabels(titles_to_plot)
ax.set_ylim(0, 1.2)
ax.set_ylabel("Score")
ax.set_title("Backend Comparison (chunk 220/50, k=3)")
ax.legend()
plt.tight_layout()
plt.savefig("ablation_backend.png", dpi=150)
plt.show()

# ── Per-type MRR breakdown ────────────────────────────────────────────────────
type_cols = [c for c in backend_df.columns if c.startswith("mrr_")]
print("\nPer-question-type MRR:")
backend_df[["backend"] + type_cols]


**How to interpret the ablation**

- Smaller chunks often improve precision because each chunk is more focused.
- Larger chunks can improve recall because they contain more context, but they may hurt ranking quality.
- Overlap is useful when important statements cross chunk boundaries.
- If `MiniLM` is available, you can rerun the ablation and compare semantic embeddings against the TF-IDF fallback.

This section gives you concrete material for the “new insights” part of the report: instead of claiming that chunking matters, you can show which configuration worked best on your paper collection and benchmark questions.


## Step 7. Pitfalls and practical extensions

**Common pitfalls**
- Extracted PDF text may contain broken lines, repeated headers, or corrupted symbols.
- A good answer can still fail if the correct paper is not retrieved in the top-k set.
- Small datasets can make the system look stronger than it really is, so it is worth adding harder questions or more papers.

**Useful extensions**
- Add metadata filters, for example by year or author.
- Store chunk embeddings on disk so repeated runs are faster.
- Replace the fallback generator with a true local instruction model.
- Add answer faithfulness checks by verifying whether every claim is supported by a retrieved chunk.


## Exercises

1. Add two new evaluation questions and rerun the ablation table.
2. Change the chunk size and inspect whether the same question retrieves different pages.
3. Switch the answer mode to `qwen_api` or `local_llm` and compare the writing quality against the extractive mode.


In [ ]:
# Exercise scaffold: write your own question here and inspect the retrieved evidence.
custom_question = "How does the EAG method differ from standard multi-agent debate?"
custom_demo = run_rag_demo(custom_question, top_k=4, answer_mode="extractive")

print("Question:", custom_demo["question"])
print()
print(textwrap.fill(custom_demo["answer"], width=110))
print()
custom_demo["retrieved_chunks"]


## Final takeaway

This notebook implements a complete lightweight RAG workflow for local academic papers:
ingestion, chunking, indexing, retrieval, answer synthesis, and a mini evaluation loop. That makes it suitable as both a working demo and a strong starting point for the Option 2 course project report.
